In [0]:
%run ./01-config

In [0]:
# Import core PySpark SQL functions (e.g., current_timestamp, from_json, window functions)
from pyspark.sql import functions as F
# Import Window for defining partitioned and ordered operations (e.g., ranking records)
from pyspark.sql.window import Window

# Construct absolute paths for raw data and streaming checkpoints
# - `landing_zone`: Where raw files (CSV/JSON) are initially staged (used in Bronze ingestion)
# - `checkpoint_base`: Directory to store streaming query offsets/state for fault tolerance
landing_zone = base_dir_data + "/raw"
checkpoint_base = base_dir_checkpoint + "/checkpoints"


# --------------------------------------------------------------------------------------------------
# GENERIC UPSERT HELPER FOR DELTA MERGE
# --------------------------------------------------------------------------------------------------

def upserter(df_micro_batch, batch_id, merge_query, temp_view):
    """
    Generic helper to execute a Delta MERGE operation on a micro-batch DataFrame.
    
    - Creates a temporary view from the micro-batch so it can be referenced in SQL MERGE.
    - Executes the provided `merge_query` using the underlying SparkSession.
    - Prints batch completion for observability.
    
    Used by most Silver upsert functions via `foreachBatch`.
    """
    df_micro_batch.createOrReplaceTempView(temp_view)
    df_micro_batch._jdf.sparkSession().sql(merge_query)
    print(f"Batch {batch_id} for {temp_view} processed.")


# --------------------------------------------------------------------------------------------------
# SPECIALIZED UPSERT FOR USER PROFILE (WITH CDC LOGIC)
# --------------------------------------------------------------------------------------------------

def upsert_user_profile_microbatch(df_micro_batch, batch_id, merge_query):
    """
    Processes a micro-batch of user profile change data (from Kafka 'user_info' topic).
    
    - Filters only 'new' and 'update' events (ignores deletes or unknown types).
    - Uses a window function to keep only the **latest update per user_id** in the batch.
    - Executes a MERGE into the Silver `user_profile` table using SQL.
    
    This ensures idempotent, up-to-date user profiles even if duplicate or out-of-order events arrive.
    """
    # Define window: rank records by `updated` timestamp descending within each `user_id`
    window = Window.partitionBy("user_id").orderBy(F.col("updated").desc())
    
    # Keep only latest 'new'/'update' record per user in this batch
    df_micro_batch = (
        df_micro_batch.filter(F.col("update_type").isin(["new", "update"]))
        .withColumn("rank", F.rank().over(window))
        .filter("rank == 1")
        .drop("rank")
    )
    
    # Create temp view for MERGE SQL
    df_micro_batch.createOrReplaceTempView("user_profile_cdc")
    
    # Execute the upsert
    df_micro_batch._jdf.sparkSession().sql(merge_query)
    
    print(f"Batch {batch_id} processed.")


# --------------------------------------------------------------------------------------------------
# SILVER LAYER UPSERT FUNCTIONS (Bronze → Silver Transformation)
# Each function reads from a Bronze Delta table and writes to a Silver Delta table using MERGE.
# --------------------------------------------------------------------------------------------------

def upsert_users(once=True, processing_time="15 seconds", startingVersion=0):
    """
    Creates/updates the `users` Silver table from `registered_users_bz` (Bronze).
    
    - Converts `registration_timestamp` from double to timestamp.
    - Uses watermarking and dropDuplicates to handle late/duplicate raw events.
    - Simple insert-only MERGE (no updates expected for user registration).
    """
    merge_query = f"""
    MERGE INTO {catalog}.{db_name}.users a
    USING users_delta b
    ON a.user_id = b.user_id
    WHEN NOT MATCHED THEN INSERT *
    """

    # Read stream from Bronze table as a change stream (Delta CDC-style)
    df_delta =(spark.readStream
    .option("startingVersion", startingVersion)      # Start from specific Delta version
    .option("ignoreDeletes", "true")                # Ignore DELETE operations in source
    .table(f"{catalog}.{db_name}.registered_users_bz")
    .selectExpr("user_id", "device_id", "mac_address", "cast(registration_timestamp as timestamp)")
    .withWatermark("registration_timestamp", "30 seconds")  # Handle late data
    .dropDuplicates(["user_id", "device_id"])       # Dedupe within watermark window
    )

    # Write using foreachBatch to enable SQL MERGE
    stream_writer =(df_delta.writeStream
        .foreachBatch(lambda df, id: upserter(df, id, merge_query, "users_delta"))
        .outputMode("update")
        .option("checkpointLocation", f"{checkpoint_base}/users")
        .queryName("users_upsert_stream")
    )

    # Trigger as batch or continuous stream
    if once == True:
        return stream_writer.trigger(availableNow=True).start()
    else:
        return stream_writer.trigger(processingTime=processing_time).start()


def upsert_gym_logs(once=True, processing_time="15 seconds", startingVersion=0):
    """
    Creates/updates the `gym_logs` Silver table from `gym_logins_bz` (Bronze).
    
    - Converts login/logout from double to timestamp.
    - Supports **partial updates**: if a newer logout time arrives, it updates the existing record.
    - Uses composite key (mac_address, gym, login) to identify sessions.
    """
    merge_query = f"""
    MERGE INTO {catalog}.{db_name}.gym_logs a
    USING gym_logs_delta b
    ON a.mac_address=b.mac_address AND a.gym=b.gym AND a.login=b.login
    WHEN MATCHED AND b.logout > a.login AND b.logout > a.logout
        THEN UPDATE SET logout = b.logout
    WHEN NOT MATCHED THEN INSERT *
    """

    df_delta = (spark.readStream
                    .option("startingVersion", startingVersion)
                    .option("ignoreDeletes", True)
                    .table(f"{catalog}.{db_name}.gym_logins_bz")
                    .selectExpr("mac_address", "gym", "cast(login as timestamp)", "cast(logout as timestamp)")
                    .withWatermark("login", "30 seconds")
                    .dropDuplicates(["mac_address", "gym", "login"])
            )

    stream_writer = (df_delta.writeStream
                                .foreachBatch(lambda df, id: upserter(df, id, merge_query, "gym_logs_delta"))
                                .outputMode("update")
                                .option("checkpointLocation", checkpoint_base + "/gyms_logs")
                                .queryName("gym_logs_upsert_stream")
                    )
    
    if once == True:
        return stream_writer.trigger(availableNow=True).start()
    else:
        return stream_writer.trigger(processingTime=processing_time).start()


def upsert_user_profile(once=True, processing_time="15 seconds", startingVersion=0):
    """
    Creates/updates the `user_profile` Silver table from Kafka 'user_info' messages in `kafka_multiplex_bz`.
    
    - Parses JSON value using defined schema (includes nested address struct).
    - Converts dob string to date, timestamp to proper timestamp.
    - Uses specialized micro-batch handler (`upsert_user_profile_microbatch`) to dedupe and filter.
    - MERGE updates only if incoming record is newer (`a.updated < b.updated`).
    """
    merge_query = f"""
    MERGE INTO {catalog}.{db_name}.user_profile a
    USING user_profile_cdc b
    ON a.user_id=b.user_id
    WHEN MATCHED AND a.updated < b.updated
        THEN UPDATE SET *
    WHEN NOT MATCHED
        THEN INSERT *
    """

    # Schema matches expected JSON structure from 'user_info' Kafka topic
    schema = """
    user_id bigint, update_type STRING, timestamp FLOAT, 
    dob STRING, sex STRING, gender STRING, first_name STRING, last_name STRING, 
    address STRUCT<street_address: STRING, city: STRING, state: STRING, zip: INT>
    """

    df_cdc = (
    spark.readStream
            .option("startingVersion", 0)           # Always start from beginning for CDC
            .option("ignoreDeletes", True)
            .table(f"{catalog}.{db_name}.kafka_multiplex_bz")
            .filter("topic = 'user_info'")
            .select(F.from_json(F.col("value").cast("string"), schema).alias("v"))
            .select("v.*")
            .select(
                "user_id",
                F.to_date("dob", "MM/dd/yyyy").alias("dob"),
                "sex", "gender", "first_name", "last_name",
                "address.*",
                F.col("timestamp").cast("timestamp").alias("updated"),
                "update_type"
            )
            .withWatermark("updated", "30 seconds")
            .dropDuplicates(["user_id", "updated"])
            )

    stream_writer = (
    df_cdc.writeStream
            .foreachBatch(lambda df, id: upsert_user_profile_microbatch(df, id, merge_query))
            .outputMode("update")
            .option("checkpointLocation", checkpoint_base + "/user_profile")
            .queryName("user_profile_stream")
             )
    
    if once == True:
        return stream_writer.trigger(availableNow=True).start()
    else:
        return stream_writer.trigger(processingTime=processing_time).start()


def upsert_workouts(once=True, processing_time="15 seconds", startingVersion=0):
    """
    Creates the `workouts` Silver table from Kafka 'workout' messages.
    
    - Parses JSON workout events (start, stop, pause, resume).
    - Converts timestamp to proper type.
    - Insert-only (no updates); duplicates removed via dropDuplicates.
    """
    merge_query = f"""
    MERGE INTO {catalog}.{db_name}.workouts a
    USING workouts_delta b
    ON a.user_id=b.user_id AND a.time=b.time
    WHEN NOT MATCHED THEN INSERT *
    """

    schema = "user_id INT, workout_id INT, timestamp FLOAT, action STRING, session_id INT"

    df_delta = (spark.readStream
            .option("startingVersion", startingVersion)
            .option("ignoreDeletes", True)
            .table(f"{catalog}.{db_name}.kafka_multiplex_bz")
            .filter("topic = 'workout'")
            .select(F.from_json(F.col("value").cast("string"), schema).alias("v"))
            .select("v.*")
            .select("user_id", "workout_id", 
                    F.col("timestamp").cast("timestamp").alias("time"), 
                    "action", "session_id")
            .withWatermark("time", "30 seconds")
            .dropDuplicates(["user_id", "time"])
    )

    stream_writer = (df_delta.writeStream
                                .foreachBatch(lambda df, id: upserter(df, id, merge_query, "workouts_delta"))
                                .outputMode("update")
                                .option("checkpointLocation",checkpoint_base + "/workouts")
                                .queryName("workouts_upsert_stream")
                    )
    
    if once == True:
        return stream_writer.trigger(availableNow=True).start()
    else:
        return stream_writer.trigger(processingTime=processing_time).start()


def upsert_heart_rate(once=True, processing_time="15 seconds", startingVersion=0):
    """
    Creates the `heart_rate` Silver table from Kafka 'bpm' messages.
    
    - Parses heart rate readings (device_id, time, heartrate).
    - Adds `valid` boolean flag: heartrate <= 0 is invalid.
    - Insert-only; duplicates removed.
    """
    merge_query = f"""
    MERGE INTO {catalog}.{db_name}.heart_rate a
    USING heart_rate_delta b
    ON a.device_id=b.device_id AND a.time=b.time
    WHEN NOT MATCHED THEN INSERT *
    """

    schema = "device_id LONG, time TIMESTAMP, heartrate DOUBLE"

    df_delta = (spark.readStream
                        .option("startingVersion", startingVersion)
                        .option("ignoreDeletes", True)
                        .table(f"{catalog}.{db_name}.kafka_multiplex_bz")
                        .filter("topic = 'bpm'")
                        .select(F.from_json(F.col("value").cast("string"), schema).alias("v"))
                        .select("v.*", F.when(F.col("v.heartrate") <= 0, False).otherwise(True).alias("valid"))
                        .withWatermark("time", "30 seconds")
                        .dropDuplicates(["device_id", "time"])
                )

    stream_writer = (df_delta.writeStream
                                .foreachBatch(lambda df, id: upserter(df, id, merge_query, "heart_rate_delta"))
                                .outputMode("update")
                                .option("checkpointLocation", checkpoint_base + "/heart_rate")
                                .queryName("heart_rate_upsert_stream")
                    )

    if once == True:
        return stream_writer.trigger(availableNow=True).start()
    else:
        return stream_writer.trigger(processingTime=processing_time).start()


# --------------------------------------------------------------------------------------------------
# USER BINNING HELPER FUNCTION
# --------------------------------------------------------------------------------------------------

from pyspark.sql.functions import floor, months_between, current_date, when, col

def age_bins(dob_col):
    """
    Converts a date-of-birth column into predefined age group bins.
    
    Used in `user_bins` table to enable demographic segmentation.
    """
    age_col = floor(months_between(current_date(), dob_col) / 12)
    
    return (when(age_col < 18, "under 18")
              .when((age_col >= 18) & (age_col < 25), "18-25")
              .when((age_col >= 25) & (age_col < 35), "25-35")
              .when((age_col >= 35) & (age_col < 45), "35-45")
              .when((age_col >= 45) & (age_col < 55), "45-55")
              .when((age_col >= 55) & (age_col < 65), "55-65")
              .when((age_col >= 65) & (age_col < 75), "65-75")
              .when((age_col >= 75) & (age_col < 85), "75-85")
              .when((age_col >= 85) & (age_col < 95), "85-95")
              .when(age_col >= 95, "95+")
              .otherwise("invalid age"))


def upsert_user_bins(once=True, processing_time="15 seconds", startingVersion=0):
    """
    Creates/updates the `user_bins` Silver table—a derived demographic summary.
    
    - Joins static `users` table (to ensure only registered users are included).
    - Reads streaming changes from `user_profile` (with `ignoreChanges=True` to get all new/updated rows).
    - Applies `age_bins()` function to categorize users by age.
    - Full upsert (update if exists, insert if new).
    """
    merge_query = f"""
    MERGE INTO {catalog}.{db_name}.user_bins a
    USING user_bins_delta b
    ON a.user_id=b.user_id
    WHEN MATCHED 
    THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
    """

    # Step 1: Load list of registered users from silver `users` table (static DataFrame)
    df_user = spark.table(f"{catalog}.{db_name}.users").select("user_id")

    # Step 2: Read streaming changes from `user_profile` table
    df_delta = (
        spark.readStream
            .option("startingVersion", startingVersion)
            .option("ignoreChanges", True)  # Get all new/changed rows (not just inserts)
            .table(f"{catalog}.{db_name}.user_profile")
            .join(df_user, on="user_id", how="left")  # Statelss left join with static DataFrame
            .select(
                "user_id",
                age_bins(col("dob")).alias("age"),  # Use previously defined function
                "gender", 
                "city", 
                "state"
            )
    )

    stream_writer = (df_delta.writeStream
                            .foreachBatch(lambda df, id: upserter(df, id, merge_query, "user_bins_delta"))
                            .outputMode("update")
                            .option("checkpointLocation", checkpoint_base + "/user_bins")
                            .queryName("user_bins_upsert_stream")
                    )

    if once == True:
        return stream_writer.trigger(availableNow=True).start()
    else:
        return stream_writer.trigger(processingTime=processing_time).start()


def upsert_completed_workouts(once=True, processing_time="15 seconds", startingVersion=0):
    """
    Derives `completed_workouts` by joining 'start' and 'stop' events from `workouts` table.
    
    - Uses stream-stream join with watermarks and state cleanup condition:
        - Stop must occur within 3 hours of start.
    - Ensures only valid, completed sessions are recorded.
    - Insert-only (idempotent by session key).
    """
    #Idempotent - Only one user workout session completes. So ignore the duplicates and insert the new records
    merge_query = f"""
    MERGE INTO {catalog}.{db_name}.completed_workouts a
    USING completed_workouts_delta b
    ON a.user_id=b.user_id AND a.workout_id = b.workout_id AND a.session_id=b.session_id
    WHEN NOT MATCHED THEN INSERT *
    """

    # Stream of 'start' events
    df_start = (spark.readStream
                        .option("startingVersion", startingVersion)
                        .option("ignoreDeletes", True)
                        .table(f"{catalog}.{db_name}.workouts")
                        .filter("action = 'start'")                         
                        .selectExpr("user_id", "workout_id", "session_id", "time as start_time")
                        .withWatermark("start_time", "30 seconds")
                )

    # Stream of 'stop' events
    df_stop = (spark.readStream
                        .option("startingVersion", startingVersion)
                        .option("ignoreDeletes", True)
                        .table(f"{catalog}.{db_name}.workouts")
                        .filter("action = 'stop'")                         
                        .selectExpr("user_id", "workout_id", "session_id", "time as end_time")
                        .withWatermark("end_time", "30 seconds")
                )

    # State cleanup - Define a condition to clean the state
    #               - stop must occur within 3 hours of start 
    #               - stop < start + 3 hours
    join_condition = [df_start.user_id == df_stop.user_id, df_start.workout_id==df_stop.workout_id, df_start.session_id==df_stop.session_id, 
                        df_stop.end_time < df_start.start_time + F.expr('interval 3 hour')]         

    df_delta = (df_start.join(df_stop, join_condition)
                        .select(df_start.user_id, df_start.workout_id, df_start.session_id, df_start.start_time, df_stop.end_time)
                )

    stream_writer = (df_delta.writeStream
                                .foreachBatch(lambda df, id: upserter(df, id, merge_query, "completed_workouts_delta"))
                                .outputMode("append")
                                .option("checkpointLocation", checkpoint_base + "/completed_workouts_delta")
                                .queryName("completed_workouts_upsert_stream")
                    )

    if once == True:
        return stream_writer.trigger(availableNow=True).start()
    else:
        return stream_writer.trigger(processingTime=processing_time).start()


def upsert_workout_bpm(once=True, processing_time="15 seconds", startingVersion=0):
    """
    Links heart rate readings to completed workout sessions.
    
    - Joins streaming `heart_rate` (valid only) with streaming `completed_workouts`.
    - Matches BPM readings that fall within a session's start/end time.
    - Uses state cleanup: session end must be within 3 hours of BPM reading.
    - Result enables per-session heart rate analysis.
    """
    #Idempotent - Only one user workout session completes. So ignore the duplicates and insert the new records
    merge_query = f"""
    MERGE INTO {catalog}.{db_name}.workout_bpm a
    USING workout_bpm_delta b
    ON a.user_id=b.user_id AND a.workout_id = b.workout_id AND a.session_id=b.session_id AND a.time=b.time
    WHEN NOT MATCHED THEN INSERT *
    """

    # Load static user table to map device_id → user_id
    df_users = spark.read.table(f"{catalog}.{db_name}.users")

    # Load the completed workouts stream (with user_id and device_id)
    df_completed_workouts = (
        spark.readStream
            .option("startingVersion", 0)
            .option("ignoreDeletes", True)
            .table(f"{catalog}.{db_name}.completed_workouts")
            .join(df_users, "user_id")
            .selectExpr("user_id", "device_id", "workout_id", "session_id", "start_time", "end_time")
            .withWatermark("end_time", "30 seconds")
    )

    # Load the heart rate stream (valid readings only)
    df_bpm = (
        spark.readStream
            .option("startingVersion", 0)
            .option("ignoreDeletes", True)
            .table(f"{catalog}.{db_name}.heart_rate")
            .filter("valid = True")
            .selectExpr("device_id", "time", "heartrate")
            .withWatermark("time", "30 seconds")
    )

    from pyspark.sql.functions import expr
    # Define the join condition between BPM and workouts
    join_condition = [
        df_completed_workouts.device_id == df_bpm.device_id,
        df_bpm.time > df_completed_workouts.start_time,
        df_bpm.time <= df_completed_workouts.end_time,
        df_completed_workouts.end_time < df_bpm.time + expr("interval 3 hours")
    ]

    # Join and select the desired columns
    df_delta = (
        df_bpm.join(df_completed_workouts, join_condition)
            .select("user_id", "workout_id", "session_id", "start_time", "end_time", "time", "heartrate")
    )

    stream_writer = (
        df_delta.writeStream
                .foreachBatch(lambda df, id: upserter(df, id, merge_query, "workout_bpm_delta"))
                .outputMode("append")
                .option("checkpointLocation", checkpoint_base + "/workout_bpm")
                .queryName("workout_bpm_upsert_stream")
    )

    if once == True:
        return stream_writer.trigger(availableNow=True).start()
    else:
        return stream_writer.trigger(processingTime=processing_time).start()


# --------------------------------------------------------------------------------------------------
# UTILITY & ORCHESTRATION FUNCTIONS
# --------------------------------------------------------------------------------------------------

def await_queries(once):
    """
    If running in batch mode (`once=True`), waits for all active streaming queries to finish.
    Ensures sequential execution of Silver layer stages.
    """
    if once:
        for stream in spark.streams.active:
            stream.awaitTermination()


def upsert_silver(once=True, processing_time="5 seconds"):
    """
    Orchestrates the full Silver layer upsert in three sequential stages:
    
    Stage 1: Core entity tables (users, gym_logs, user_profile, workouts, heart_rate)
    Stage 2: Derived tables requiring Stage 1 (user_bins, completed_workouts)
    Stage 3: Session-level aggregations (workout_bpm)
    
    Waits for each stage to complete before starting the next (when `once=True`).
    """
    import time
    start = int(time.time())
    print(f"\n Executing silver layer upsert ...")

    # Silver Layer 1: Core profile & device-level streams
    upsert_users(once, processing_time)
    upsert_gym_logs(once, processing_time)
    upsert_user_profile(once, processing_time)
    upsert_workouts(once, processing_time)
    upsert_heart_rate(once, processing_time)

    await_queries(once)
    print(f"Completed silver layer 1 upsert in {int(time.time()) - start} seconds")

    # Silver Layer 2: Derived user binning & session logs
    upsert_user_bins(once, processing_time)
    upsert_completed_workouts(once, processing_time)

    await_queries(once)
    print(f" Completed silver layer 2 upsert in {int(time.time()) - start} seconds")

    # Silver Layer 3: Workout BPM aggregation
    upsert_workout_bpm(once, processing_time)

    await_queries(once)
    print(f" Completed silver layer 3 upsert in {int(time.time()) - start} seconds")


# --------------------------------------------------------------------------------------------------
# VALIDATION FUNCTIONS
# --------------------------------------------------------------------------------------------------

def assert_count(catalog, db_name, table_name, expected_count, filter="true"):
    """
    Validates exact row count in a table (optionally with a SQL filter).
    Used to confirm data quality after Silver layer processing.
    """
    print(f"Validating record counts in {table_name}...", end='')
    actual_count = spark.read.table(f"{catalog}.{db_name}.{table_name}").where(filter).count()
    assert actual_count == expected_count, f"Expected {expected_count:,} records, found {actual_count:,} in {table_name} where {filter}"
    print(f"Found {actual_count:,} / Expected {expected_count:,} records where {filter}: Success")


def validate_silver(catalog, db_name, sets):
    """
    Validates record counts across all Silver tables.
    
    - `sets=1`: Expected counts after one batch of test data.
    - `sets=2`: Expected counts after two batches (e.g., re-ingestion).
    
    Validates in three stages to match the upsert order.
    """
    import time
    start = int(time.time())
    print(f"\n Starting Silver layer validation...")

    # Silver layer 1
    assert_count(catalog, db_name, "users", 5 if sets == 1 else 10)
    assert_count(catalog, db_name, "gym_logs", 8 if sets == 1 else 16)
    assert_count(catalog, db_name, "user_profile", 5 if sets == 1 else 10)
    assert_count(catalog, db_name, "workouts", 16 if sets == 1 else 32)
    assert_count(catalog, db_name, "heart_rate", sets * 253801)

    print(f"Silver layer 1 validation done in {int(time.time()) - start} seconds\n")

    # Silver layer 2
    assert_count(catalog, db_name, "user_bins", 5 if sets == 1 else 10)
    assert_count(catalog, db_name, "completed_workouts", 8 if sets == 1 else 16)

    print(f"Silver layer 2 validation done in {int(time.time()) - start} seconds\n")

    # Silver layer 3
    assert_count(catalog, db_name, "workout_bpm", 3968 if sets == 1 else 8192)

    print(f"Silver layer 3 validation done in {int(time.time()) - start} seconds ")